# 01 · Exploratory data analysis
Corrected CICIDS2017 (Liu, Engelen et al., CNS 2022). Raw class distribution, data-quality issues, and feature correlations **before** any cleaning.

In [ ]:
%cd -q ..
# notebooks run from the project root so relative config paths resolve

In [ ]:
import logging
import pandas as pd
from IPython.display import Image, display

from ids import experiments as ex
from ids.config import load_config, results_dir

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.max_columns", 50)

# Use configs/smoke.yaml to try a notebook on synthetic data (python -m ids.synthetic first)
cfg = load_config("configs/default.yaml")
OUT = results_dir(cfg)

In [ ]:
from ids import datasets
raw = datasets.load(cfg.dataset.name, cfg.dataset.dir, cfg.dataset.attempted_as_benign)
print(raw.shape)
raw.groupby(["day", "label"]).size().rename("flows").to_frame()

### Class imbalance
The log scale is the point: Benign is several orders of magnitude larger than Heartbleed/Infiltration.

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
counts = raw["family"].value_counts()
ax = counts.plot.barh(logx=True, figsize=(7, 4)); ax.set_xlabel("flows (log scale)"); plt.show()
(counts / counts.sum()).round(6).to_frame("share")

### 'Attempted' flows
The corrected dataset labels flows that belong to an attack session but carry no attack payload as `... - Attempted`. Following the dataset authors, these are treated as benign.

In [ ]:
raw[raw["label"].str.contains("Attempted")]["label"].value_counts()

### Data quality: missing / infinite values and duplicates

In [ ]:
import numpy as np
feats = datasets.feature_columns(raw)
X = raw[feats].to_numpy(np.float32)
print("rows with NaN/inf:", int((~np.isfinite(X)).any(axis=1).sum()))
print("exact duplicate rows:", int(raw[feats + ["family"]].duplicated().sum()))
print("constant features:", [c for c in feats if raw[c].nunique() <= 1])

### Feature correlation
Many CICFlowMeter features are near-copies of each other (e.g. subflow counts vs totals). Phase 3 prunes pairs above |r| = 0.95.

In [ ]:
sample = raw.sample(min(len(raw), 200_000), random_state=cfg.seed)[feats]
corr = sample.corr().abs()
plt.figure(figsize=(14, 12)); sns.heatmap(corr, cmap="viridis", xticklabels=False); plt.show()
high = (corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().sort_values(ascending=False))
high[high > 0.95].head(30)